In [ ]:
"""
Candidate Ranking Pipeline v2
Hybrid BM25 + Semantic Embedding retrieval with CrossEncoder reranking.
Fixes: punctuation in tokens, hyphenated metadata keywords, score compression.
"""

import os
import sys
import re
import json
import pickle
import zipfile
import xml.etree.ElementTree as ET
import logging

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

# ---------------------------------------------------------------------------
# LOGGING
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
DATASET_DIR = "/content/India_runs_data_and_ai_challenge_extracted/India_runs_data_and_ai_challenge"
CANDIDATES_PATH = f"{DATASET_DIR}/candidates.jsonl"
JOB_DESCRIPTION_PATH = f"{DATASET_DIR}/job_description.docx"

PROCESSED_DIR = "/content/data/processed"
ARTIFACTS_DIR = "/content/data/artifacts"
ENRICHED_CANDIDATES_PATH = f"{PROCESSED_DIR}/enriched_candidates.jsonl"
BM25_INDEX_PATH = f"{ARTIFACTS_DIR}/bm25.pkl"
EMBEDDINGS_PATH = f"{ARTIFACTS_DIR}/embeddings.npy"
CANDIDATE_IDS_PATH = f"{ARTIFACTS_DIR}/candidate_ids.json"
OUTPUT_SUBMISSION_PATH = "/content/submission.csv"

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 100        # candidates in final submission
RERANK_POOL = 200  # broader pool passed to cross-encoder before final cut
EMBED_BATCH_SIZE = 64  # reduce to 32 if OOM on CPU

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
log.info(f"Using device: {device}")


# ---------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------

def clean_tokenize(text: str) -> list:
    """
    Lowercase, strip all punctuation (including brackets and trailing commas),
    expand hyphenated compounds into separate tokens, then split on whitespace.
    Prevents artifacts like '(sentence-transformers,' becoming a BM25 token.
    """
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)   # remove ALL non-word, non-space chars
    return [t for t in text.split() if t]


def extract_text_from_docx(file_path: str) -> str:
    """
    Extract plain text from a .docx file.
    Raises RuntimeError on failure so the caller can decide how to handle it.
    """
    try:
        with zipfile.ZipFile(file_path) as docx:
            xml_content = docx.read("word/document.xml")
            tree = ET.fromstring(xml_content)
            ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
            texts = [node.text for node in tree.findall(".//w:t", ns) if node.text]
            result = " ".join(texts).strip()
            if not result:
                raise ValueError("Extracted JD text is empty.")
            return result
    except Exception as e:
        raise RuntimeError(f"Failed to parse JD docx at '{file_path}': {e}") from e


def safe_join(*parts) -> str:
    """Join string parts, skipping None / non-string values."""
    return " ".join(str(p) for p in parts if p and isinstance(p, str)).strip()


def process_candidate(candidate: dict):
    """
    Flatten a candidate record into a single enriched text string.
    Returns None if the record has no usable content.
    """
    candidate_id = candidate.get("candidate_id")
    if not candidate_id:
        return None

    profile = candidate.get("profile") or {}
    career = candidate.get("career_history") or []
    skills = candidate.get("skills") or []
    education = candidate.get("education") or []

    profile_text = safe_join(
        profile.get("headline"),
        profile.get("summary"),
        profile.get("location"),
    )

    career_parts = []
    for role in career:
        if not isinstance(role, dict):
            continue
        career_parts.append(
            safe_join(role.get("title"), role.get("company"), role.get("description"))
        )
    career_text = " ".join(career_parts)

    skill_parts = []
    for s in skills:
        if isinstance(s, dict):
            skill_parts.append(s.get("name", ""))
        elif isinstance(s, str):
            skill_parts.append(s)
    skills_text = " ".join(filter(None, skill_parts))

    edu_parts = []
    for e in education:
        if isinstance(e, dict):
            edu_parts.append(safe_join(e.get("degree"), e.get("institution")))
    edu_text = " ".join(edu_parts)

    enriched_text = safe_join(profile_text, career_text, skills_text, edu_text)
    if not enriched_text:
        log.warning(f"Candidate {candidate_id} produced empty enriched text; skipping.")
        return None

    return {"candidate_id": candidate_id, "enriched_text": enriched_text}


def min_max_normalize(arr: np.ndarray) -> np.ndarray:
    rng = arr.max() - arr.min()
    return (arr - arr.min()) / (rng + 1e-10)


# ---------------------------------------------------------------------------
# PHASE 1: ENRICH CANDIDATES
# ---------------------------------------------------------------------------
log.info("Phase 1: Enriching candidates...")

seen_ids = set()
enriched_count = 0
skipped_count = 0

with open(ENRICHED_CANDIDATES_PATH, "w") as outfile, open(CANDIDATES_PATH, "r") as infile:
    for line_no, line in enumerate(infile, 1):
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as e:
            log.warning(f"Line {line_no}: JSON parse error — skipping. ({e})")
            skipped_count += 1
            continue

        enriched = process_candidate(record)
        if enriched is None:
            skipped_count += 1
            continue

        cid = enriched["candidate_id"]
        if cid in seen_ids:
            log.warning(f"Duplicate candidate_id '{cid}' on line {line_no} — skipping.")
            skipped_count += 1
            continue

        seen_ids.add(cid)
        outfile.write(json.dumps(enriched) + "\n")
        enriched_count += 1

log.info(f"Enrichment complete: {enriched_count} saved, {skipped_count} skipped.")

# ---------------------------------------------------------------------------
# PHASE 2: EMBEDDINGS
# ---------------------------------------------------------------------------
log.info("Phase 2: Computing embeddings...")

texts = []
cids = []

with open(ENRICHED_CANDIDATES_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        texts.append(rec["enriched_text"])
        cids.append(rec["candidate_id"])

with open(CANDIDATE_IDS_PATH, "w") as f:
    json.dump(cids, f)

log.info(f"Loaded {len(texts)} candidates. Encoding with '{EMBEDDING_MODEL_NAME}'...")
bi_encoder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

embeddings = bi_encoder.encode(
    texts,
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # unit-norm → dot product == cosine similarity
)
np.save(EMBEDDINGS_PATH, embeddings)
log.info(f"Embeddings saved: shape={embeddings.shape}")

# ---------------------------------------------------------------------------
# PHASE 3: JD EXTRACTION + BM25 INDEX
# ---------------------------------------------------------------------------
log.info("Phase 3: Extracting JD and building BM25 index...")

try:
    jd_text = extract_text_from_docx(JOB_DESCRIPTION_PATH)
    log.info(f"JD extracted ({len(jd_text)} chars). Preview: {jd_text[:200]!r}")
except RuntimeError as e:
    log.error(str(e))
    sys.exit(1)

# BM25 — use clean_tokenize so corpus matches query tokenization exactly
corpus = [clean_tokenize(t) for t in texts]
bm25 = BM25Okapi(corpus)
with open(BM25_INDEX_PATH, "wb") as f:
    pickle.dump(bm25, f)
log.info(f"BM25 index saved to {BM25_INDEX_PATH}.")

# ---------------------------------------------------------------------------
# PHASE 4: HYBRID RETRIEVAL
# ---------------------------------------------------------------------------
log.info("Phase 4: Hybrid ranking (BM25 + cosine similarity)...")

jd_vec = bi_encoder.encode([jd_text], normalize_embeddings=True, convert_to_numpy=True)[0]
cos_sim = embeddings @ jd_vec           # cosine sim (unit vectors)
bm25_scores = bm25.get_scores(clean_tokenize(jd_text))

cos_norm = min_max_normalize(cos_sim)
bm25_norm = min_max_normalize(bm25_scores)

# Weighted hybrid: semantic signal dominates for AI/ML roles
hybrid_scores = 0.6 * cos_norm + 0.4 * bm25_norm

pool_size = min(RERANK_POOL, len(hybrid_scores))
pool_idx = np.argsort(-hybrid_scores)[:pool_size]
log.info(f"Top-{pool_size} pool selected for cross-encoder reranking.")

# ---------------------------------------------------------------------------
# PHASE 5: CROSS-ENCODER RERANKING
# ---------------------------------------------------------------------------
log.info("Phase 5: Cross-encoder reranking...")

cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL_NAME, device=device)
pairs = [(jd_text, texts[i]) for i in pool_idx]
ce_scores = cross_encoder.predict(pairs, show_progress_bar=True)

ce_global_scores = np.full(len(texts), -np.inf)
for local_rank, global_idx in enumerate(pool_idx):
    ce_global_scores[global_idx] = ce_scores[local_rank]

final_top_idx = np.argsort(-ce_global_scores)[:TOP_K]

# ---------------------------------------------------------------------------
# PHASE 6: DYNAMIC REASONING + SUBMISSION
# ---------------------------------------------------------------------------
log.info("Phase 6: Building submission with dynamic reasoning...")

# Extract clean JD keywords using the same tokenizer as BM25
STOPWORDS = {
    "the", "and", "for", "with", "to", "of", "in", "a", "an", "is",
    "are", "was", "were", "be", "been", "has", "have", "had", "on",
    "at", "by", "from", "as", "or", "not", "this", "that", "will",
    "can", "our", "you", "your", "their", "we", "us", "who", "which",
    "must", "should", "also", "work", "role", "team", "strong", "using",
    "use", "used", "experience", "ability", "skills", "good", "well",
}
jd_tokens_clean = clean_tokenize(jd_text)
jd_keywords = sorted(
    {
        w for w in jd_tokens_clean
        if len(w) > 4          # skip very short tokens
        and w not in STOPWORDS
        and w.isalpha()        # pure alpha only — drops numeric and punctuation artifacts
    },
    key=len,
    reverse=True,
)[:30]
log.info(f"Top JD keywords: {jd_keywords[:15]}")


def generate_reasoning(candidate_text: str, cos_score: float, bm25_score: float) -> str:
    """Generate a candidate-specific, clean reasoning string."""
    candidate_tokens = set(clean_tokenize(candidate_text))
    matched = [kw for kw in jd_keywords if kw in candidate_tokens][:5]
    keyword_str = (
        f"Matched skills/keywords: {', '.join(matched)}."
        if matched
        else "General ML/AI profile alignment."
    )
    sem_level = "strong" if cos_score > 0.75 else "moderate" if cos_score > 0.5 else "partial"
    bm25_level = "high" if bm25_score > 0.75 else "moderate" if bm25_score > 0.5 else "low"
    return (
        f"{keyword_str} "
        f"Semantic fit: {sem_level} ({cos_score:.2f}). "
        f"Keyword coverage: {bm25_level} ({bm25_score:.2f})."
    )


submission_rows = []
for rank, idx in enumerate(final_top_idx, 1):
    submission_rows.append({
        "candidate_id": cids[idx],
        "rank": rank,
        "score": round(float(ce_global_scores[idx]), 4),
        "reasoning": generate_reasoning(texts[idx], cos_norm[idx], bm25_norm[idx]),
    })

df = pd.DataFrame(submission_rows, columns=["candidate_id", "rank", "score", "reasoning"])

# Single normalization pass on cross-encoder scores
df["score"] = min_max_normalize(df["score"].values).round(4)

df.to_csv(OUTPUT_SUBMISSION_PATH, index=False)
log.info(f"Submission saved to {OUTPUT_SUBMISSION_PATH} ({len(df)} rows).")

try:
    display(df.head(10))  # type: ignore[name-defined]  # Jupyter
except NameError:
    print(df.head(10).to_string(index=False))